# Fine-Tuning TinyLlama with LoRA — Beginner Notebook
### Built for: Shruti Agrawal | Role prep: Accenture LLM Developer

---

## What are we doing here?

We are taking a small pre-trained language model called **TinyLlama** and teaching it to behave like a **customer support assistant** using LoRA fine-tuning.

Think of it like this:
- TinyLlama is a person who has read the entire internet (pre-trained)
- But it doesn't know how to be a polite, structured customer support agent
- We are going to train it on examples of good customer support conversations
- After training, it will respond in that style

## Before you run anything — do this first:
1. Click **Runtime** in the top menu
2. Click **Change Runtime Type**
3. Select **T4 GPU** under Hardware Accelerator
4. Click Save

This gives you a free GPU so training doesn't take forever.

---

## CELL 1 — Install the tools we need

Just like installing apps on your phone before using them, we need to install Python libraries first.

Here is what each library does in plain English:
- **transformers** — HuggingFace's main library. Lets us download and use pre-trained models
- **peft** — The library that contains LoRA. PEFT = Parameter Efficient Fine-Tuning
- **trl** — Makes the training process easier. Has a ready-made trainer for fine-tuning
- **datasets** — Lets us load and prepare training data easily
- **accelerate** — Helps the training run faster and use GPU properly
- **bitsandbytes** — Allows us to compress the model to use less memory (quantization)

This cell will take about 2-3 minutes. You will see a lot of text scroll by — that is normal.

In [ ]:
# The ! at the start means: run this as a terminal command, not Python
# -q means quiet mode — installs without showing every single detail

!pip install -q transformers peft trl datasets accelerate bitsandbytes

## CELL 2 — Import the tools into our notebook

Installing puts the tools on the computer. Importing brings them INTO our current session so we can use them.

Think of installing as buying a tool, and importing as picking it up from the shelf to use it.

Here is what each import does:
- **AutoModelForCausalLM** — Loads the TinyLlama model. 'Causal LM' means it generates text left to right (like GPT style)
- **AutoTokenizer** — Converts our text into numbers the model understands (and back again)
- **BitsAndBytesConfig** — Settings for compressing the model to 4-bit so it fits in free Colab memory
- **LoraConfig** — Where we set our LoRA settings (rank, alpha etc.)
- **get_peft_model** — Takes our base model and wraps LoRA around it
- **SFTTrainer** — The trainer that handles the fine-tuning loop for us automatically
- **TrainingArguments** — Settings for training (how many steps, learning rate etc.)
- **load_dataset** — Loads our training data
- **torch** — PyTorch. The engine that runs all the maths underneath

In [ ]:
# Each 'from X import Y' means: from library X, bring in tool Y

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset, Dataset
import torch

# Let's confirm our GPU is available
# If this prints 'cuda', you have a GPU. If it prints 'cpu', go back and set GPU in Runtime settings
print("Device available:", "cuda" if torch.cuda.is_available() else "cpu")

## CELL 3 — Create our training data

This is the most important cell to understand — because **this is your actual job skill**.

Fine-tuning needs **instruction-response pairs** — examples of what the model should learn.
Each pair has:
- An **instruction** (the user's message)
- A **response** (what we want the model to say)

We format it in a special template called **ChatML** format that TinyLlama understands.
This is the same concept as the annotation and structuring work you did at NVIDIA —
you were creating structured input-output pairs for models to learn from.
That work is THIS step.

In a real project at Accenture, this data would come from a client's support logs,
internal knowledge base, or domain documents. Here we create a small sample manually.

In [ ]:
# These are our training examples
# Each example teaches the model one customer support interaction
# The more examples you have, the better the model learns
# In real projects you would have thousands — just like your 13,000+ annotations

training_examples = [
    {
        "instruction": "My order has not arrived yet. It has been 10 days.",
        "response": "I apologize for the delay. Could you please share your order ID so I can check the status for you? Standard delivery takes 5-7 business days, but I will investigate this right away."
    },
    {
        "instruction": "I want to return a product I bought last week.",
        "response": "Of course! Our return window is 30 days from purchase. Please visit your Orders section, select the item, and click 'Return'. You will receive a prepaid label via email within 24 hours."
    },
    {
        "instruction": "How do I reset my account password?",
        "response": "To reset your password, click 'Forgot Password' on the login page and enter your registered email. You will receive a reset link within a few minutes. Please also check your spam folder if you don't see it."
    },
    {
        "instruction": "I was charged twice for the same order.",
        "response": "I sincerely apologize for this inconvenience. This should not have happened. Please share your order ID and the last 4 digits of the payment method used, and I will initiate a refund for the duplicate charge immediately."
    },
    {
        "instruction": "Can I change my delivery address after placing an order?",
        "response": "Address changes are possible within 1 hour of placing the order. If it has been less than an hour, please contact us immediately with your order ID. After 1 hour, the order goes to our warehouse and cannot be redirected."
    },
    {
        "instruction": "My product arrived damaged.",
        "response": "I am truly sorry to hear that. Please take a photo of the damaged item and packaging, then email it to support@store.com with your order ID. We will arrange a replacement or full refund within 2 business days."
    },
    {
        "instruction": "What are your customer support working hours?",
        "response": "Our customer support team is available Monday to Friday, 9 AM to 6 PM IST. For urgent issues outside these hours, you can email us and we will respond within 24 hours on the next business day."
    },
    {
        "instruction": "I never received my refund.",
        "response": "Refunds typically take 5-7 business days to reflect in your account depending on your bank. Could you share your refund confirmation ID? I will check the exact status and escalate if there is any delay on our end."
    },
    {
        "instruction": "How do I cancel my subscription?",
        "response": "To cancel your subscription, go to Account Settings, click on Subscriptions, and select Cancel. Your access will continue until the end of the current billing period. Would you like me to walk you through any specific step?"
    },
    {
        "instruction": "Is there a discount available for students?",
        "response": "Yes! We offer a 20% student discount through our education program. You can apply at our website under the Students section by verifying your enrollment with a valid college email address or student ID."
    },
]

# Now we format each example into the ChatML template
# This is the exact format TinyLlama was trained on, so it understands it
# <|system|> tells the model its role
# <|user|> is the customer's message
# <|assistant|> is what we want the model to learn to say

def format_example(example):
    # This function takes one instruction-response pair and formats it properly
    formatted = f"""<|system|>
You are a helpful, polite, and professional customer support assistant. Always be empathetic and solution-focused.</s>
<|user|>
{example['instruction']}</s>
<|assistant|>
{example['response']}</s>"""
    return {"text": formatted}

# Apply the formatting function to all our examples
# This creates a list of formatted training texts
formatted_data = [format_example(ex) for ex in training_examples]

# Convert our list into a HuggingFace Dataset object
# The trainer needs data in this specific format
dataset = Dataset.from_list(formatted_data)

# Let's check what one example looks like after formatting
print("Sample training example:")
print("-" * 50)
print(dataset[0]["text"])
print("-" * 50)
print(f"\nTotal training examples: {len(dataset)}")

## CELL 4 — Load the model and tokenizer

Now we download TinyLlama from HuggingFace. This is around 2GB so it takes a few minutes.

We also load it in **4-bit quantization** — this means we compress the model to use
less memory so it fits on Colab's free GPU. Quality is almost the same, size is much smaller.

The **tokenizer** converts text into numbers (tokens) that the model reads.
For example: "Hello" → [15043] and [15043] → "Hello" on the way back out.

In [ ]:
# The model name on HuggingFace Hub
# HuggingFace is like a library — this is the book's title and author
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# BitsAndBytesConfig sets up 4-bit quantization
# load_in_4bit=True: compress model weights to 4-bit numbers (saves ~75% memory)
# bnb_4bit_quant_type="nf4": the specific compression method (NormalFloat4 — best quality)
# bnb_4bit_use_double_quant=True: compresses even the compression settings (saves a tiny bit more)
# bnb_4bit_compute_dtype=torch.float16: while stored in 4bit, compute in 16bit for stability

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("Downloading TinyLlama... (this takes 2-4 minutes)")

# Load the model from HuggingFace with our quantization settings
# device_map="auto": automatically puts the model on GPU if available
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# Load the tokenizer for the same model
tokenizer = AutoTokenizer.from_pretrained(model_name)

# pad_token tells the tokenizer what to use for padding short sequences
# to the same length as longer ones (needed during batch training)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Prepare the model for LoRA training
# This function from PEFT makes the quantized model compatible with training
model = prepare_model_for_kbit_training(model)

print("\nModel loaded successfully!")
print(f"Model is on: {next(model.parameters()).device}")

## CELL 5 — Set up LoRA

This is the most important cell — this is where we actually apply LoRA to the model.

Remember what LoRA does: instead of training all 1.1 billion parameters of TinyLlama,
we freeze the whole model and add tiny trainable matrices to specific layers.
Only these tiny matrices will learn during our training.

**The settings explained in plain English:**
- **r (rank)** — The size of our tiny LoRA matrices. Bigger = learns more but uses more memory. 8 or 16 is good for beginners
- **lora_alpha** — How strongly the LoRA updates affect the model. Rule of thumb: set it to 2x your rank
- **lora_dropout** — Randomly switches off some LoRA neurons during training to prevent overfitting (memorizing instead of learning)
- **target_modules** — Which specific layers inside the model to apply LoRA to. q_proj and v_proj are the attention layers — most important ones to fine-tune
- **task_type** — CAUSAL_LM means we are doing text generation (predict next word style)

In [ ]:
# LoraConfig holds all our LoRA settings
lora_config = LoraConfig(
    r=8,                          # rank — size of the small matrices we add
    lora_alpha=16,                # scaling factor — how much influence LoRA has
    lora_dropout=0.05,            # 5% dropout to prevent overfitting
    target_modules=[              # which layers to apply LoRA to
        "q_proj",                 # query projection in attention
        "v_proj"                  # value projection in attention
    ],
    bias="none",                  # don't train bias parameters
    task_type=TaskType.CAUSAL_LM  # we are doing text generation
)

# Apply LoRA to the model
# get_peft_model wraps our base model with the LoRA layers on top
model = get_peft_model(model, lora_config)

# print_trainable_parameters shows us HOW FEW parameters we are actually training
# This is the power of LoRA — instead of training 1.1B params, we train a tiny fraction
model.print_trainable_parameters()

## CELL 6 — Set training settings and train the model

Now we tell the trainer HOW to train — how many times to go through the data,
how fast to learn, where to save the results.

**Training settings explained:**
- **num_train_epochs** — How many times the model sees all the training data. 3 is enough for small datasets
- **per_device_train_batch_size** — How many examples to process at once. Keep at 2 for free Colab GPU
- **learning_rate** — How big each learning step is. Too high = unstable. Too low = learns nothing. 2e-4 is safe
- **max_seq_length** — Maximum length of text in tokens. 512 covers all our examples comfortably
- **output_dir** — Where to save the fine-tuned model weights

**This cell will take about 5-10 minutes to complete on a T4 GPU.**
You will see a progress bar. Watch the 'loss' number — it should go down over time.
Decreasing loss = the model is learning.

In [ ]:
# SFTConfig holds all our training settings
# SFT = Supervised Fine-Tuning
training_args = SFTConfig(
    output_dir="./tinyllama-customer-support",  # folder to save the trained model
    num_train_epochs=3,                          # go through all data 3 times
    per_device_train_batch_size=2,               # 2 examples at a time (safe for free GPU)
    gradient_accumulation_steps=4,              # pretend we used batch size 8 (2x4)
    learning_rate=2e-4,                          # step size for learning (0.0002)
    fp16=True,                                   # use 16-bit precision for speed
    logging_steps=5,                             # print loss every 5 steps
    save_steps=50,                               # save model checkpoint every 50 steps
    max_seq_length=512,                          # maximum token length per example
    dataset_text_field="text",                   # which column in our dataset has the text
    report_to="none"                             # don't send logs to wandb (keeps it simple)
)

# SFTTrainer handles the entire training loop for us
# It takes: the model, training settings, training data, and tokenizer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer
)

print("Starting training...")
print("Watch the 'loss' value — it should decrease over time.")
print("Lower loss = model is learning better.")
print("-" * 50)

# This single line runs the entire training process
trainer.train()

print("-" * 50)
print("Training complete!")

## CELL 7 — Save the fine-tuned model

After training, we save the LoRA weights.

Note: we only save the LoRA adapter weights — the tiny matrices we trained.
NOT the full TinyLlama model. This is one of the benefits of LoRA —
the saved file is very small (a few MB) instead of the full 2GB model.

In [ ]:
# Save only the LoRA adapter weights (not the full base model)
# This is why LoRA is so efficient — the adapter is tiny
trainer.model.save_pretrained("./tinyllama-customer-support-final")
tokenizer.save_pretrained("./tinyllama-customer-support-final")

print("Model saved to: ./tinyllama-customer-support-final")
print("\nNote: Only the LoRA adapter weights are saved, not the full base model.")
print("This is one of the key advantages of LoRA fine-tuning.")

## CELL 8 — Test our fine-tuned model

Now the exciting part — let's talk to our model and see if it learned to be a customer support assistant!

We'll ask it questions and compare how it responds. If the loss went down during training,
it should respond in a polite, structured customer support style.

In [ ]:
# Function to generate a response from our fine-tuned model
def get_response(user_message, max_new_tokens=200):
    """
    Takes a user message and returns the model's response.
    
    Steps happening inside:
    1. Format the message in ChatML template (same as training)
    2. Tokenize it (convert text to numbers)
    3. Pass to model and generate response tokens
    4. Decode the tokens back to text
    5. Return just the assistant's part
    """
    
    # Format the prompt the same way we formatted training data
    prompt = f"""<|system|>
You are a helpful, polite, and professional customer support assistant. Always be empathetic and solution-focused.</s>
<|user|>
{user_message}</s>
<|assistant|>
"""
    
    # Tokenize — convert text to numbers the model reads
    # return_tensors="pt" means return PyTorch tensors
    # .to(model.device) puts the input on the same device as the model (GPU)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate response
    # do_sample=True: model picks from probable next words (more natural)
    # temperature=0.7: controls randomness. Lower = more focused, Higher = more creative
    # max_new_tokens: maximum words in the response
    with torch.no_grad():  # no_grad saves memory during inference (we are not training here)
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode — convert numbers back to text
    # skip_special_tokens=True: removes the <|system|>, <|user|> tags from output
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract just the assistant's response (everything after the last 'assistant' marker)
    if "assistant" in full_response.lower():
        response_parts = full_response.split("<|assistant|>")
        if len(response_parts) > 1:
            return response_parts[-1].strip()
    
    return full_response.strip()


# Test it with questions the model has NOT seen during training
test_questions = [
    "I want to track my order.",
    "How do I update my billing information?",
    "My account has been locked."
]

print("Testing fine-tuned model responses:")
print("=" * 60)

for question in test_questions:
    print(f"\nCustomer: {question}")
    print(f"Assistant: {get_response(question)}")
    print("-" * 60)

## CELL 9 — Try it yourself!

Now try asking your own questions to the fine-tuned model.
Change the text inside the quotes and run the cell.

In [ ]:
# Change this message to anything you want to ask!
your_question = "I received the wrong item in my order."

print(f"Customer: {your_question}")
print(f"\nAssistant: {get_response(your_question)}")

## Congratulations — You just fine-tuned an LLM with LoRA!

---

## What you did in this notebook — in interview language:

1. **Designed an instruction-response dataset** for supervised fine-tuning (Cell 3)
2. **Loaded a pre-trained LLM** (TinyLlama 1.1B) with 4-bit quantization (Cell 4)
3. **Applied LoRA** using HuggingFace PEFT with rank=8 on attention layers (Cell 5)
4. **Fine-tuned the model** using SFTTrainer with QLoRA on a domain-specific task (Cell 6)
5. **Saved the LoRA adapter weights** separately from the base model (Cell 7)
6. **Evaluated the model** by testing on unseen customer queries (Cell 8)

## How to answer the interview question: 'Walk me through a fine-tuning project'

*"I fine-tuned TinyLlama 1.1B using QLoRA and HuggingFace PEFT for a customer support use case.
I designed a 10-example instruction-response dataset formatted in ChatML template,
applied LoRA with rank 8 on the attention projection layers,
and trained using SFTTrainer for 3 epochs with a learning rate of 2e-4.
The trainable parameters were under 1% of the full model,
and the model learned to respond in a structured, empathetic support style.
I evaluated it on held-out queries to check generalization."*

---

## What to try next to go deeper:
- Add more training examples (aim for 50-100)
- Try increasing rank to r=16 and see if quality improves
- Change the domain — try a medical FAQ assistant or an HR policy bot
- Try a different base model: Phi-2 or Qwen2.5-0.5B
- Upload your adapter to HuggingFace Hub and share the link on your LinkedIn